# EIB: Six-Country Respondent History Windows (c3110)

**Task:** same as the static baseline (predict each respondent's **c3110** answer, 1 = much worse … 5 = much
better), but the prompt now also shows the respondent's **own earlier answers to c3110**. We vary how many
earlier answers are shown (**history window = 0 / 3 / 6 / 12**) and ask how much history helps.

**Built on `six_country_static_baseline.ipynb`:** same files, same evaluation rows (re-drawn with the same
seed and checked against that run), same model, system prompt, persona fields and scores. Window 0 is exactly
the static prompt, so the static baseline is re-run here under identical conditions.

**History rule** (from the team's history file `six_country_history_up_to12_2020_2026.csv`): only waves strictly
before the target wave; at most the most recent N valid answers, oldest
first, with the gap shown as "k months earlier" (no calendar dates, so the model gets no macro hint from the
date). Earlier answers need not be consecutive months. Respondents with fewer than N earlier answers get all
they have (history is optional; with none, the prompt equals the static one).

**Comparisons:**
- window 0 vs 3 / 6 / 12 on the **same rows** (main table: rows with ≥ 12 earlier answers, so every window is full);
- each LLM window vs a **no-LLM history rule** with the same information (last answer; answer shares over the
  last N): history only adds value through the LLM if the LLM beats these;
- rows with no earlier answer get identical prompts in every window, so their score spread is run-to-run noise.

**Macro context (Model 1):** `MACRO_TEXT` is the hook for the country-level macro block; with it set, the same
notebook runs static + macro + history (Model 2).

**Before you start:** Colab: add `ANTHROPIC_API_KEY` in 🔑 Secrets. Locally: put `ANTHROPIC_API_KEY=...` in
`history_window/.env`.

## 0. Settings

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DATA_DIR = '/content/drive/MyDrive/EIB_ECB2'
    OUT_DIR = '/content/drive/MyDrive/EIB_runs/six_country_history_window'
    RESULTS_DIR = OUT_DIR
else:
    DATA_DIR = 'data'                  # relative to history_window/; respondent-level, gitignored
    OUT_DIR = 'data/runs'              # answer caches and per-row predictions
    RESULTS_DIR = 'results'            # aggregate tables only
STATIC_FILE = 'six_country_static_persona_baseline_2020_2026.csv'
TARGET_FILE = 'six_country_c3110_labels_evaluation_only_2020_2026.csv'
HISTORY_FILE = 'six_country_history_up_to12_2020_2026.csv'

COUNTRIES = ['BE', 'DE', 'ES', 'FR', 'IT', 'NL']
TARGET = 'c3110'
VALID = [1, 2, 3, 4, 5]
OPTIONS = {1: 'get a lot worse', 2: 'get a little worse', 3: 'stay about the same',
           4: 'get a little better', 5: 'get a lot better'}
QUESTION = ('Looking ahead to 12 months from now, do you expect the financial situation of your household '
            'to be better or worse than it is now?')
PERSONA_FIELDS = {
    'gender': 'gender_released',
    'age group': 'age_group',
    'education': 'education_group',
    'household net income quintile (1 = lowest 20% in country, 5 = highest)': 'income_quintile',
    'household size': 'household_size_topcoded',
    'children aged 0-17 in household': 'children_0_17_topcoded',
    'lives with partner': 'partner_in_household',
}

N_PER_COUNTRY = 200
HISTORY_CUTS = [3, 6, 12]
SEED = 42
WINDOWS = [0, 3, 6, 12]      # 0 = static prompt
MAIN_SUBSET = 12             # main comparison: rows with at least this many earlier answers
N_BOOT = 2000

MODEL = 'claude-haiku-5-5'
EFFORT = 'low'
MAX_CONCURRENCY = 8

MACRO_TEXT = None            # Model 1 hook: function(row) -> text block, inserted after the profile

# The static baseline run (2026-10-27): its first two sampled rows per country (printed in API completion order)
# and its history-subset sizes.
# Re-drawing the sample must reproduce them, otherwise these rows are not the baseline's rows.
BASELINE_TEST_ROWS = ['p_45697196_w14', 'p_64822670_w58', 'p_83153814_w37', 'p_47248092_w33',
                      'p_60619146_w76', 'p_20599389_w39', 'p_34943365_w41', 'p_90180310_w57',
                      'p_75321020_w61', 'p_90034727_w21', 'p_43086071_w59', 'p_39588240_w38']
BASELINE_SUBSET_SIZES = {3: 918, 6: 730, 12: 459}

## 1. Setup

In [ ]:
import os, json, time, asyncio
import numpy as np, pandas as pd
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    !pip -q install -U anthropic scikit-learn scipy
elif os.path.exists('.env'):
    for line in open('.env'):
        k, _, v = line.strip().partition('=')
        if k and not k.startswith('#'):
            os.environ.setdefault(k.strip(), v.strip().strip('"\''))
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)

## 2. Load the static persona file and the c3110 answers, join one-to-one

Same checks as the static baseline: unique keys, one-to-one join, country and month agree, valid answers only.

In [ ]:
static = pd.read_csv(f'{DATA_DIR}/{STATIC_FILE}', low_memory=False)
assert not static.duplicated(['a0010', 'a0030']).any(), 'duplicated keys in static file'
assert set(static.a0020) <= set(COUNTRIES), f'unexpected countries: {set(static.a0020) - set(COUNTRIES)}'
assert not [c for c in static.columns if c.lower().startswith(('c1', 'c3'))], 'survey answers in static file'
exp_year, exp_month = 2020 + (static.a0030 - 1) // 12, (static.a0030 - 1) % 12 + 1
assert ((static.survey_year == exp_year) & (static.survey_month == exp_month)).all(), 'wave/month mismatch'

ans = pd.read_csv(f'{DATA_DIR}/{TARGET_FILE}', low_memory=False)
assert (ans['target_variable'] == TARGET).all()
ans = ans[ans.target_valid.astype(bool)].rename(columns={'target_value': TARGET})
check_cols = [c for c in ['a0020', 'country', 'date', 'survey_period'] if c in ans.columns]
ans_small = ans[['a0010', 'a0030', TARGET] + check_cols].rename(columns={c: c + '_ans' for c in check_cols})
merged = static.merge(ans_small, on=['a0010', 'a0030'], how='outer', validate='one_to_one', indicator=True)
print(merged['_merge'].value_counts().to_dict())
df = merged[merged._merge == 'both'].drop(columns='_merge').copy()
for c in check_cols:
    assert (df[c].astype(str) == df[c + '_ans'].astype(str)).all(), f'{c} disagrees between the files'

df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')
df = df[df[TARGET].isin(VALID)].copy(); df[TARGET] = df[TARGET].astype(int)
df = df.sort_values(['a0010', 'a0030'])
df['n_prior'] = df.groupby('a0010').cumcount()
print(f'{len(df):,} usable rows, {df.a0010.nunique():,} people, waves {df.a0030.min()}-{df.a0030.max()}')

## 3. Evaluation sample: the static baseline's rows

Same draw as the static baseline (seed 42, ≤ 1 row per person, 200 per country). If that run's sample file is
available it is used directly; either way the rows are checked against the baseline run.

In [ ]:
SAMPLE_PATH = f'{OUT_DIR}/eval_sample_seed{SEED}_n{N_PER_COUNTRY}.csv'
if os.path.exists(SAMPLE_PATH):
    keys = pd.read_csv(SAMPLE_PATH, usecols=['a0010', 'a0030'])
    sample = keys.merge(df, on=['a0010', 'a0030'], how='left', validate='one_to_one')
else:
    sample = (df.sample(frac=1, random_state=SEED).drop_duplicates('a0010')
                .groupby('a0020', group_keys=False).head(N_PER_COUNTRY).reset_index(drop=True))
    sample.to_csv(SAMPLE_PATH, index=False)
sample['row_id'] = sample.a0010.astype(str) + '_w' + sample.a0030.astype(str)

test_ids = (sample.groupby('a0020', group_keys=False).head(2)).row_id.tolist()
assert set(test_ids) == set(BASELINE_TEST_ROWS), 'sample differs from the static baseline run'
sizes = {k: int((sample.n_prior >= k).sum()) for k in HISTORY_CUTS}
assert sizes == BASELINE_SUBSET_SIZES, f'history-subset sizes differ from the baseline run: {sizes}'
print('Same rows as the static baseline:', sample.groupby('a0020').size().to_dict(), '|', sizes)

## 4. Respondent history (the team's history file)

`six_country_history_up_to12_2020_2026.csv` holds, for every respondent-wave, the person's last ≤ 12 valid c3110
answers in strictly earlier waves (`c3110_history{k}_value` / `_wave`, k = 1 most recent). It is joined on
a0010 + a0030 and checked against the answers file: same number of earlier answers, same waves and values.

In [ ]:
MAX_HIST = 12
hcols = [f'c3110_history{k}_{f}' for k in range(1, MAX_HIST + 1) for f in ('value', 'wave')]
hist = pd.read_csv(f'{DATA_DIR}/{HISTORY_FILE}', usecols=['a0010', 'a0030', 'a0020', 'n_prior_valid_c3110'] + hcols)
hist = sample[['row_id', 'a0010', 'a0030', 'a0020', 'n_prior']].merge(
    hist, on=['a0010', 'a0030'], how='left', validate='one_to_one', suffixes=('', '_hist'))
assert (hist.a0020 == hist.a0020_hist).all(), 'country disagrees between the history file and the sample'
assert (hist.n_prior_valid_c3110 == hist.n_prior).all(), 'history file counts earlier answers differently'

HIST = {}                                    # row_id -> rows of (wave, answer), oldest first
for _, r in hist.iterrows():
    k = min(int(r.n_prior), MAX_HIST)
    HIST[r.row_id] = np.array([[r[f'c3110_history{j}_wave'], r[f'c3110_history{j}_value']] for j in range(k, 0, -1)],
                              dtype=float).astype(int).reshape(-1, 2)

past = df.loc[df.a0010.isin(sample.a0010), ['a0010', 'a0030', TARGET]]
by_person = {p: g[['a0030', TARGET]].to_numpy() for p, g in past.groupby('a0010')}
for _, r in sample.iterrows():
    h = by_person[r.a0010]
    assert np.array_equal(h[h[:, 0] < r.a0030][-MAX_HIST:], HIST[r.row_id]), f'history mismatch: {r.row_id}'
print('history file agrees with the answers file on all', len(HIST), 'rows')
sample['last_answer'] = [HIST[r][-1, 1] if len(HIST[r]) else np.nan for r in sample.row_id]
print('earlier answers per row:', sample.n_prior.describe()[['min', '50%', 'max']].to_dict())
print('true answer = last answer (rows with history):',
      round((sample[TARGET] == sample.last_answer)[sample.n_prior > 0].mean(), 3))

## 5. Prompt

Window 0 gives exactly the static baseline prompt. A window N adds the most recent N earlier answers.

In [ ]:
COUNTRY_NAMES = {'BE': 'Belgium', 'DE': 'Germany', 'ES': 'Spain', 'FR': 'France', 'IT': 'Italy', 'NL': 'the Netherlands'}
assert TARGET not in PERSONA_FIELDS.values()

SYSTEM = ('You simulate how a specific respondent of the ECB Consumer Expectations Survey answers. '
          'Predict what this particular person would answer, not what is economically correct. '
          'Give calibrated probabilities that reflect your real uncertainty about this person.')

def persona_text(row):
    lines = [f'- country: {COUNTRY_NAMES[row["a0020"]]}']
    for name, col in PERSONA_FIELDS.items():
        v = row[col]
        if isinstance(v, float) and v.is_integer(): v = int(v)
        if pd.notna(v): lines.append(f'- {name}: {str(v).replace(" (released recode)", "")}')
    return '\n'.join(lines)

def history_text(row, window):
    h = HIST[row['row_id']][-window:] if window else HIST[row['row_id']][:0]
    if not len(h):
        return None
    lines = []
    for wave, a in h:
        m = int(row['a0030'] - wave)
        lines.append(f'- {m} month{"s" if m > 1 else ""} earlier: {int(a)} ({OPTIONS[int(a)]})')
    return 'Earlier answers of this respondent to the same question (oldest first):\n' + '\n'.join(lines)

def build_prompt(row, window):
    opts = '\n'.join(f'  {k}: {v}' for k, v in OPTIONS.items())
    blocks = [f'Respondent profile:\n{persona_text(row)}']
    if MACRO_TEXT is not None:
        blocks.append(MACRO_TEXT(row))
    hist = history_text(row, window)
    if hist:
        blocks.append(hist)
    return ('\n\n'.join(blocks) + f'\n\nQuestion: {QUESTION}\nOptions:\n{opts}\n\n'
            'Give the probability that this respondent picks each option (keys "1".."5", summing to 1).')

ex = sample[sample.n_prior >= 12].iloc[0]
print(SYSTEM, '\n--- window 0 ---\n' + build_prompt(ex, 0), '\n--- window 6 ---\n' + build_prompt(ex, 6))

## 6. LLM calls (same request as the static baseline; one cache file per window, so a crash resumes)

In [ ]:
import anthropic
client = anthropic.AsyncAnthropic()
SCHEMA = {'type': 'object',
          'properties': {'probs': {'type': 'object',
                                   'properties': {str(k): {'type': 'number'} for k in VALID},
                                   'required': [str(k) for k in VALID], 'additionalProperties': False}},
          'required': ['probs'], 'additionalProperties': False}

async def ask(row, window, sem):
    async with sem:
        for attempt in range(5):
            try:
                resp = await client.beta.messages.create(
                    model=MODEL, max_tokens=2000, system=SYSTEM,
                    messages=[{'role': 'user', 'content': build_prompt(row, window)}],
                    output_config={'effort': EFFORT, 'format': {'type': 'json_schema', 'schema': SCHEMA}},
                    betas=['server-side-fallback-2026-07-01'], fallbacks='default',
                )
                if resp.stop_reason == 'refusal':
                    return {'row_id': row['row_id'], 'error': 'refusal'}
                out = json.loads(next(b.text for b in resp.content if b.type == 'text'))
                return {'row_id': row['row_id'], 'probs': out['probs'], 'model': resp.model,
                        'in_tok': resp.usage.input_tokens, 'out_tok': resp.usage.output_tokens}
            except (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError):
                await asyncio.sleep(5 * 2 ** attempt)
            except Exception as e:
                return {'row_id': row['row_id'], 'error': repr(e)[:300]}
        return {'row_id': row['row_id'], 'error': 'retries exhausted'}

def cache_path(tag): return f'{OUT_DIR}/answers_{tag}_{MODEL}.jsonl'

def load_answers(path):
    recs = [json.loads(l) for l in open(path)]
    ok = {r['row_id']: r for r in recs if 'error' not in r}
    errs = {r['row_id']: r['error'] for r in recs if 'error' in r and r['row_id'] not in ok}
    if errs: print(f'{len(errs)} rows failed, e.g. {next(iter(errs.values()))} (re-run the cell to retry)')
    rows = []
    for r in ok.values():
        p = np.array([max(float(r['probs'][str(k)]), 0.0) for k in VALID])
        p = p / p.sum() if p.sum() > 0 else np.full(len(VALID), 1 / len(VALID))
        rows.append({'row_id': r['row_id'], **{f'p{k}': p[i] for i, k in enumerate(VALID)}, 'pred': VALID[int(p.argmax())]})
    return pd.DataFrame(rows)

def usage(path):
    recs = [json.loads(l) for l in open(path)]
    return sum(r.get('in_tok', 0) for r in recs), sum(r.get('out_tok', 0) for r in recs)

async def run_rows(rows, window, tag):
    path = cache_path(tag)
    done = {json.loads(l)['row_id'] for l in open(path) if '"error"' not in l} if os.path.exists(path) else set()
    todo = [r for _, r in rows.iterrows() if r['row_id'] not in done]
    print(f'[{tag}] {len(done)} cached, {len(todo)} to call ({MODEL}, effort={EFFORT})')
    sem = asyncio.Semaphore(MAX_CONCURRENCY); t0 = time.time()
    tasks = [asyncio.create_task(ask(r, window, sem)) for r in todo]
    with open(path, 'a') as f:
        for i, fut in enumerate(asyncio.as_completed(tasks), 1):
            res = await fut; f.write(json.dumps(res) + '\n'); f.flush()
            if i % 300 == 0 or i == len(tasks): print(f'  {i}/{len(tasks)}  {time.time()-t0:.0f}s')
    return load_answers(path)

### 6b. Test on 12 rows with long histories (check the output before the full run)

In [ ]:
test_rows = sample[sample.n_prior >= 12].groupby('a0020', group_keys=False).head(2)
test_out = {}
for w in WINDOWS:
    test_out[w] = await run_rows(test_rows, w, f'test_w{w}')
pd.concat({f'w{w}': o.set_index('row_id')['pred'] for w, o in test_out.items()}, axis=1).join(
    test_rows.set_index('row_id')[['a0020', 'last_answer', TARGET]])

## 7. Full run: every window on all 1,200 rows

In [ ]:
answers = {}
for w in WINDOWS:
    answers[w] = await run_rows(sample, w, f'full_w{w}')
print({w: len(a) for w, a in answers.items()})

## 8. Reference rules (no LLM)

All use only answers from **earlier waves** than the row being predicted.
- **country_mode**, **persona_cell**: as in the static baseline.
- **last_answer**: the person's most recent earlier answer (falls back to persona_cell without history).
- **history_share_N**: the person's answer shares over their last N earlier answers; prediction = most common,
  ties go to the most recent (falls back to persona_cell without history). Same information as LLM window N.

In [ ]:
K = len(VALID)

def shares(s):
    return s.value_counts(normalize=True).reindex(VALID, fill_value=0).values

def to_frame(row_ids, P, pred=None):
    P = np.asarray(P, dtype=float)
    out = pd.DataFrame(P, columns=[f'p{k}' for k in VALID]); out.insert(0, 'row_id', list(row_ids))
    out['pred'] = np.array(VALID)[P.argmax(axis=1)] if pred is None else pred
    return out

def prior_shares(row, cols, min_n=30):
    mask = (df.a0030 < row['a0030'])
    for c in cols: mask &= (df[c] == row[c])
    p = df.loc[mask, TARGET]
    return shares(p) if len(p) >= min_n else None

def history_share(rid, n, fallback):
    h = HIST[rid][-n:, 1]
    if not len(h):
        return fallback, VALID[int(np.argmax(fallback))]
    p = np.bincount(h.astype(int), minlength=K + 1)[1:] / len(h)
    top = [k for k in VALID if p[k - 1] == p.max()]
    pred = next(int(a) for a in h[::-1] if a in top)       # tie -> most recent
    return p, pred

cm, cell = [], []
for _, r in sample.iterrows():
    c = prior_shares(r, ['a0020'], 1)
    c = c if c is not None else shares(df.loc[df.a0020 == r['a0020'], TARGET])
    cm.append(c)
    z = prior_shares(r, ['a0020', 'age_group', 'income_quintile'])
    cell.append(z if z is not None else c)
RULES = {'country_mode': to_frame(sample.row_id, cm), 'persona_cell': to_frame(sample.row_id, cell)}
for name, n in [('last_answer', 1)] + [(f'history_share_{n}', n) for n in WINDOWS if n]:
    ps = [history_share(rid, n, fb) for rid, fb in zip(sample.row_id, cell)]
    RULES[name] = to_frame(sample.row_id, [p for p, _ in ps], [k for _, k in ps])

## 9. Scores

Same scores as the static baseline. Higher is better: accuracy, macro-F1. Lower is better: Brier, JS divergence
(predicted vs actual answer distribution, per country, averaged). Entropy ratio / option coverage: 1.0 = as
spread out as the real answers.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy

def score(pred, s):
    m = s.merge(pred, on='row_id', how='inner')
    y, yhat = m[TARGET].values, m['pred'].values
    P = m[[f'p{k}' for k in VALID]].values; Y = np.eye(K)[y - 1]
    js = np.mean([jensenshannon(shares(g[TARGET]), shares(g['pred']), base=2) for _, g in m.groupby('a0020')])
    return {'n': len(m), 'accuracy': accuracy_score(y, yhat), 'f1_macro': f1_score(y, yhat, average='macro'),
            'brier': np.mean(np.sum((P - Y) ** 2, axis=1)), 'js_divergence': js,
            'entropy_ratio': entropy(shares(pd.Series(yhat))) / entropy(shares(pd.Series(y))),
            'option_coverage': len(set(yhat)) / len(set(y))}

LLM = {f'LLM window {w}': a for w, a in answers.items()}
ALL = {**LLM, **RULES}
subsets = {f'>= {MAIN_SUBSET} prior (main)': sample[sample.n_prior >= MAIN_SUBSET],
           'all': sample,
           **{f'>= {k} prior': sample[sample.n_prior >= k] for k in HISTORY_CUTS if k != MAIN_SUBSET},
           'no prior (noise check)': sample[sample.n_prior == 0]}
summary = pd.concat({sub: pd.DataFrame({name: score(p, s) for name, p in ALL.items()}).T
                     for sub, s in subsets.items()})
summary.round(3)

### 9b. Paired differences on the main subset (bootstrap 95% CI over rows)

Accuracy: positive = first model better. Brier: negative = first model better.

In [ ]:
def per_row(pred, s):
    m = s[['row_id', TARGET]].merge(pred, on='row_id', how='inner')
    P = m[[f'p{k}' for k in VALID]].values; Y = np.eye(K)[m[TARGET].values - 1]
    return pd.DataFrame({'correct': (m['pred'] == m[TARGET]).astype(float).values,
                         'brier': np.sum((P - Y) ** 2, axis=1)}, index=m.row_id)

def paired(a, b, s, rng=np.random.default_rng(SEED)):
    A, B = per_row(ALL[a], s), per_row(ALL[b], s)
    ids = A.index.intersection(B.index); d = (A.loc[ids] - B.loc[ids]).values
    boot = d[rng.integers(0, len(d), (N_BOOT, len(d)))].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return {'n': len(ids), 'd_accuracy': d[:, 0].mean(), 'acc_ci': f'[{lo[0]:+.3f}, {hi[0]:+.3f}]',
            'd_brier': d[:, 1].mean(), 'brier_ci': f'[{lo[1]:+.3f}, {hi[1]:+.3f}]'}

main = subsets[f'>= {MAIN_SUBSET} prior (main)']
pairs = [(f'LLM window {w}', 'LLM window 0') for w in WINDOWS if w] + \
        [(f'LLM window {w}', f'history_share_{w}') for w in WINDOWS if w] + \
        [(f'LLM window {w}', 'last_answer') for w in WINDOWS if w]
diffs = pd.DataFrame({f'{a}  vs  {b}': paired(a, b, main) for a, b in pairs}).T
diffs.round(3)

### 9c. Copying or reading the history?

On the main subset: how often each prediction equals the last earlier answer, and accuracy separately on rows
where the true answer **stayed** equal to the last answer vs **changed**.

In [ ]:
m0 = main[['row_id', TARGET, 'last_answer']]
stay = m0[TARGET] == m0.last_answer
copying = {}
for name in list(LLM) + ['last_answer'] + [f'history_share_{w}' for w in WINDOWS if w]:
    m = m0.merge(ALL[name][['row_id', 'pred']], on='row_id')
    st = m[TARGET] == m.last_answer
    copying[name] = {'pred = last answer': (m.pred == m.last_answer).mean(),
                  f'acc | stayed (n={int(st.sum())})': (m.pred == m[TARGET])[st].mean(),
                  f'acc | changed (n={int((~st).sum())})': (m.pred == m[TARGET])[~st].mean()}
copying = pd.DataFrame(copying).T
copying.round(3)

In [ ]:
per_country = pd.concat({c: pd.DataFrame({name: score(p, main[main.a0020 == c]) for name, p in ALL.items()}).T
                         for c in COUNTRIES})
per_country[['n', 'accuracy', 'f1_macro', 'brier']].round(3)

In [ ]:
dist = pd.DataFrame({'actual': shares(main[TARGET]),
                     **{name: shares(main[['row_id']].merge(p, on='row_id')['pred']) for name, p in ALL.items()}},
                    index=[f'{k}: {v}' for k, v in OPTIONS.items()])
dist.round(3)

## 10. Save

In [ ]:
stamp = time.strftime('%Y%m%d_%H%M')
tok = {f'w{w}': usage(cache_path(f'full_w{w}')) for w in WINDOWS}
summary.to_csv(f'{RESULTS_DIR}/summary.csv'); diffs.to_csv(f'{RESULTS_DIR}/paired_differences.csv')
copying.to_csv(f'{RESULTS_DIR}/copy_vs_change.csv'); per_country.to_csv(f'{RESULTS_DIR}/per_country_main.csv')
dist.to_csv(f'{RESULTS_DIR}/distribution_main.csv')
preds = sample[['row_id', 'a0010', 'a0030', 'a0020', 'n_prior', 'last_answer', TARGET]]
for w, a in answers.items():
    preds = preds.merge(a.rename(columns=lambda c: c if c == 'row_id' else f'w{w}_{c}'), on='row_id', how='left')
preds.to_csv(f'{OUT_DIR}/llm_predictions_{stamp}.csv', index=False)
json.dump({'model': MODEL, 'effort': EFFORT, 'windows': WINDOWS, 'main_subset_min_prior': MAIN_SUBSET,
           'countries': COUNTRIES, 'target': TARGET, 'persona_fields': PERSONA_FIELDS, 'seed': SEED,
           'static_file': STATIC_FILE, 'target_file': TARGET_FILE, 'history_file': HISTORY_FILE, 'system_prompt': SYSTEM,
           'macro_block': MACRO_TEXT is not None, 'tokens_in_out_per_window': tok, 'run': stamp},
          open(f'{RESULTS_DIR}/config.json', 'w'), indent=2)
print('Saved', sorted(os.listdir(RESULTS_DIR)), '| tokens in/out:', tok)